# DTCR-U-Net: Kaggle Training & Evaluation Pipeline
This notebook trains the **Dual-Task Consistency Regularized U-Net (DTCR-U-Net)** on Kaggle GPU.

### Step 1: Ensure Kaggle GPU is Enabled
Go to **Settings** (right-hand panel) -> **Accelerator** -> **GPU T4 x2** or **P100**.

In [ ]:
# Clone repo or pull latest updates
!git clone https://github.com/shadyOg/fyp-dtcr-unet.git || (cd fyp-dtcr-unet && git pull origin main)
%cd fyp-dtcr-unet/dtcr-unet
!pip install -q -r requirements.txt

### Step 1.5: Discover Input Datasets
Shows what datasets are attached and auto-detects MosMedData and LIDC-IDRI paths.

In [ ]:
import os
from pathlib import Path

print('=== Kaggle Input Datasets ===')
input_root = Path('/kaggle/input')
for d in sorted(input_root.iterdir()):
    n_files = sum(1 for _ in d.rglob('*') if _.is_file())
    print(f'  {d.name}/ ({n_files} files)')

# Auto-detect MosMedData path (contains study_*.nii files)
MOSMED_PATH = None
for d in input_root.iterdir():
    nii_files = list(d.rglob('study_*.nii*'))
    if nii_files:
        MOSMED_PATH = str(d)
        print(f'\n\u2705 MosMedData found at: {MOSMED_PATH} ({len(nii_files)} NIfTI files)')
        break
if not MOSMED_PATH:
    print('\n\u274c MosMedData NOT found in /kaggle/input/')

# Auto-detect LIDC-IDRI path (contains seg_LIDC-IDRI dir or LIDC-IDRI-* patient dirs)
LIDC_PATH = None
for d in input_root.iterdir():
    seg_dirs = list(d.rglob('seg_LIDC-IDRI'))
    if seg_dirs:
        LIDC_PATH = str(d)
        n_slices = sum(1 for _ in seg_dirs[0].rglob('slice.png'))
        print(f'\u2705 LIDC-IDRI found at: {LIDC_PATH} ({n_slices} slices in {seg_dirs[0]})')
        break
if not LIDC_PATH:
    for d in input_root.iterdir():
        lidc_dirs = list(d.rglob('LIDC-IDRI-*'))
        if lidc_dirs:
            LIDC_PATH = str(d)
            print(f'\u2705 LIDC-IDRI found at: {LIDC_PATH} ({len(lidc_dirs)} patient directories)')
            break
if not LIDC_PATH:
    print('\u274c LIDC-IDRI NOT found in /kaggle/input/ -- add it as a dataset input')

PROCESSED_OUT = '/kaggle/working/data/processed'
print(f'\nOutput directory: {PROCESSED_OUT}')

### Step 2: Run Data Preprocessing
Extracts 2D slices, performs HU windowing, CLAHE, Level Set Map computation ($T(y)$), and creates patient-level train/val/test splits.

Processes **both** MosMedData and LIDC-IDRI if available.

In [ ]:
# Build preprocessing command with auto-detected paths
cmd = f'python data/preprocessing.py --out {PROCESSED_OUT} --size 256 --context 1 --labeled-ratio 0.4'

if MOSMED_PATH:
    cmd += f' --mosmed {MOSMED_PATH}'
if LIDC_PATH:
    cmd += f' --lidc {LIDC_PATH}'

print(f'Running: {cmd}\n')
!{cmd}

### Step 3: Train DTCR-U-Net (Dual-Task Semi-Supervised with AMP)
Runs 80 epochs with AdamW, Cosine Annealing, AMP mixed precision (FP16), and dynamic consistency weight $\lambda_d(t)$.

In [ ]:
!python train.py \
    --data {PROCESSED_OUT} \
    --epochs 80 \
    --batch-size 8 \
    --grad-accum 2 \
    --lr 0.001 \
    --labeled-ratio 0.4 \
    --checkpoints /kaggle/working/checkpoints \
    --outputs /kaggle/working/outputs

### Step 4: Evaluate Model on Test Set
Computes Dice, Sensitivity (SE), Specificity (SP), Accuracy, F1, and Hausdorff Distance (HD) on the test set, and saves visual segmentation comparisons.

In [ ]:
!python evaluate.py \
    --checkpoint /kaggle/working/checkpoints/best_model.pth \
    --data {PROCESSED_OUT} \
    --out /kaggle/working/outputs/evaluation \
    --batch-size 8